# 11/05 수업 코드 참고 패키지: 메인 LangGraph + 검색 Tool

v2.0에서 오늘 할 일은 **기존 검색 자산을 Tool로 래핑하고 메인 Graph에서 한 번 호출해 근거 응답까지 연결**하는 것이다.
이 노트북은 두 수업 폴더의 실제 코드와, 프로젝트에 맞춰 새로 작성한 실행 예시를 함께 제공한다.

1. 수업 원본: Markdown 코드 블록, 출처·셀·해시 표시. 원본 DB/API에 의존하므로 실행 셀로 복사하지 않는다.
2. 계약 원문: 같은 패키지의 v2.0 계약 §9에서 복사한 정적 타입. 실행 가능한 코드 셀로 제공한다.
3. 프로젝트 예시: 가상의 메모리 DB와 Tool을 사용한 독립 실행 셀. **공식 지침·실DB 근거가 아니다.**
4. 실제 LangGraph: 마지막에 설치된 langgraph의 StateGraph를 실행한다. API·Neo4j·.env·원본 수업 파일 없이 동작한다.

외부 읽기/쓰기·API 호출·인덱스 생성·적재는 하지 않는다. 실제 서비스 구현은 오늘 부품 지시서의 경로와 검증을 따른다.
고급 재검색 루프는 11/06 범위다. 오늘 예시는 한 호출 후 충분/부분/근거없음/오류를 구분해 종료한다.


## D01. 문서·역할·조건을 검사하는 관문

`40-7 check`는 RANGE, blocks_of, block_all, ends, COMPANY_MARK를 참조한다.
입력은 회사/관계/근거 행 사전, 출력은 (판정, 이유)다. 아래 관련 보조 함수도 포함했다.
학교 적용에서는 수업 회사 규칙을 가져오는 대신 **참조 존재·원문·역할·조건을 분리해 검사하는 방식**을 가져온다.
v2.0 Tool 반환 직후 구조 검사와 DB 확인을 하고, 의미 검수는 골든셋/사람 검수로 남긴다.


**수업 원본 — `block_all`**

`40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 전체 셀 31(1부터). SHA-256 `3dbeacd21bb76f6db9f3992b4e9d04147eaffffb5d47150e9fd0604bee88d393`

```python
def block_all(block):
    """블록의 글자 전부 - 표면 칸 글자를 이어 붙임"""
    if block["kind"] == "text":
        return block["text"]
    return " ".join(" ".join(row) for row in block["rows"])
```


**수업 원본 — `ends`**

`40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 전체 셀 32(1부터). SHA-256 `7c2ff14fd9c505f71504e02215a82ce10c28f9b3dab31f13194dd124418f8250`

```python
def ends(r):
    """대조할 이름들 - 양 끝(@제출회사 빼고) + 품목"""
    items = [x for x in str(r["item"]).split(", ") if x]
    return [x for x in [r["subject"], r["object"]] + items if x and x != "@제출회사"]
```


**수업 원본 — `check`**

`40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 전체 셀 34(1부터). SHA-256 `befdaedd545f88eec9c8622d43315ebc3205641ebbae7bf6047666cbbc8a8ecc`

```python
def check(r):
    """정답 없이 하는 검사 -> (판정, 이유). 위에서부터 처음 걸린 것 하나"""
    if r["subject"] == "" or r["object"] == "" or r["relation"] not in RANGE:
        return "위반", "빈 칸 · 허용 안 된 관계"
    if r["subject"] == r["object"]:
        return "위반", "주어와 목적어가 같음"                     # 품목을 item 칸에 잘못 넣은 경우가 많음
    block = blocks_of.get((r["rcept_no"], r["block_id"]))
    if block is None:
        return "위반", "근거 블록이 그 보고서에 없음"
    text = block_all(block)
    if not all(name in text for name in ends(r)):
        return "확인 필요", "양 끝 · 품목이 근거 블록에 없음"
    parts = [p.strip() for p in str(r["evidence"]).split("|") if p.strip()]   # 근거 글 조각들
    if len(parts) == 0 or not all(p in text for p in parts):
        return "확인 필요", "근거 글이 원문과 다름"
    if r["relation"] == "공급" and ("연결실체" in text or "계약" in text):
        return "확인 필요", "연결실체 · 계약 언급 - 원문 확인"
    if r["relation"] in ("생산", "매입") and COMPANY_MARK.search(r["object"]):
        return "확인 필요", "품목 자리에 법인 표기"
    if block["kind"] == "table" and r["row"] < 1:
        return "확인 불가", "표의 어느 행인지 하나로 못 정함"
    if block["kind"] == "table" and not all(p in " ".join(block["rows"][r["row"]]) for p in parts):
        return "확인 필요", "근거 글이 찾은 행과 다름"
    return "통과", ""
```


## D02. 청크 검색과 출처 formatter

`38-1 by_text` 의존성: driver, fulltext chunk_text, Result.to_df, Chunk→PART_OF→Article.
`38-2 with_source` 의존성: RetrieverResultItem, record의 node/score와 수업의 id 형식.
학교에서는 HAS_CHUNK로 문서 소속을 확인하고 chunk_id·문서·쪽수·원문을 구조적으로 보존한다.
Article.id.split('#') 규칙은 우리 ID에 쓰지 않는다. fulltext는 선택된 Tool 내부 전략이며 Action 직접 탐색의 의무 선행 단계가 아니다.


**수업 원본 — `by_text`**

`38-1_벡터검색과-전문검색-Vector-FullText인덱스_배포용.ipynb`, 전체 셀 38(1부터). SHA-256 `6bf9899f1fba77596ac4423bbf755ac5ed1ead9128078998773dab789f870192`

```python
def by_text(query, k=3):
    """전문 검색 상위 k Chunk 의 조문 - 구문으로 찾으려면 query 를 큰따옴표로 감싸 넘긴다"""
    return driver.execute_query("""
        CALL db.index.fulltext.queryNodes('chunk_text', $w) YIELD node, score
        MATCH (node)-[:PART_OF]->(a:Article)
        RETURN a.id AS article, a.title AS title, round(score, 2) AS score
        LIMIT $k
    """, w=query, k=k, result_transformer_=Result.to_df)
```


**수업 원본 — `with_source`**

`38-2_리트리버-Vector와-Hybrid검색_배포용.ipynb`, 전체 셀 27(1부터). SHA-256 `deb5126542ae41d4d7a17247ff7a53dfdde653a536d18562a0e482942634cea5`

```python
def with_source(record):
    node = record["node"]
    article = node["id"].split("#")[0]
    return RetrieverResultItem(content=f"[{article}] {node['text']}",
                               metadata={"article": article, "chunk": node["id"], "score": record["score"]})
```


## D03. 그래프 확장과 고정 질의 도구

`EXPAND`는 검색기가 주입한 node/score에서 Article·DELEGATES 경로를 찾는다.
expanded_item은 조회한 필드를 RetrieverResultItem으로 바꾼다. driver·벡터/fulltext 인덱스·embedder는 바깥에서 준비한다.
`37-6 rows`는 driver의 records를 사전 목록으로 바꾸는 작은 어댑터다.
학교 Action은 이미 있는 get_role_actions/get_action_evidence 같은 매개변수 Cypher를 먼저 재사용한다.
CrossDoc은 같은 ActionConcept의 출처별 행동을 비교하되 후보를 같은 규칙으로 병합하지 않는다.


**수업 원본 — `EXPAND =`**

`38-3_그래프로-근거넓히기-VectorCypher와-PageRank재랭킹_배포용.ipynb`, 전체 셀 10(1부터). SHA-256 `4b0b57164148a402bb324183fcead3d3050772df5907d766c4050d83b066212e`

```python
from neo4j_graphrag.retrievers import VectorCypherRetriever, HybridCypherRetriever, HybridRetriever
from neo4j_graphrag.types import RetrieverResultItem

EXPAND = """
MATCH (node)-[:PART_OF]->(a:Article)
OPTIONAL MATCH path = (a)-[:DELEGATES*1..2]->(target)
WITH node, score, a, collect(DISTINCT [n IN nodes(path) | coalesce(n.id, n.name)]) AS paths
RETURN a.id AS article, a.title AS title, node.text AS text, score, paths,
       COUNT { MATCH (a)<-[:REFERS]-(c:Case) RETURN DISTINCT c } AS cases, a.pagerank AS pagerank
"""
print("준비 완료")
```


**수업 원본 — `expanded_item`**

`38-3_그래프로-근거넓히기-VectorCypher와-PageRank재랭킹_배포용.ipynb`, 전체 셀 11(1부터). SHA-256 `aa9a10907e5cc65ed5e4652965809e17f6b831f8c26a0e60399c91f70593ad1e`

```python
def expanded_item(record):
    paths = [" -> ".join(path) for path in record["paths"] if path]
    content = f"[{record['article']}] {record['title']}: {record['text'][:300]}\n위임 경로: {'; '.join(paths) or '없음'}"
    return RetrieverResultItem(content=content, metadata={"article": record["article"], "score": record["score"],
                                                          "pagerank": record["pagerank"], "cases": record["cases"]})
```


**수업 원본 — `rows`**

`37-6_공종별-공공공사-시장을-묻는-GraphRAG_배포용.ipynb`, 전체 셀 21(1부터). SHA-256 `b4ae6f7926f0bcf1a02f5e0b42c816f58c3abefad9847de8da03aee14511b301`

```python
def rows(query, **params):
    """질의 결과를 사전 목록으로"""
    return [record.data() for record in driver.execute_query(query, **params).records]
```


**수업 원본 — `top_leaders`**

`37-6_공종별-공공공사-시장을-묻는-GraphRAG_배포용.ipynb`, 전체 셀 25(1부터). SHA-256 `6892e9132bfb62f33d6d46d5c80bba3a7cf116559214cd2db2e4c9794e6977ce`

```python
@tool
def top_leaders(work_type: WorkType, region: str = "") -> str:
    """한 공종에서 주계약업체(컨소시엄 대표)로 많이 나온 회사 5곳을 찾는다. region 에 '경기' · '서울' 같은
    지역 낱말을 넣으면 수요기관 이름에 그 낱말이 든 계약만 본다. '소방 공사를 주로 이끄는 회사는' 같은 질문에 쓴다."""
    found = rows("""MATCH (co:Company)-[:JOINED {role: '주계약업체'}]->(c:Contract {work_type: $w})
        WHERE $r = '' OR c.demand CONTAINS $r
        WITH co, count(c) AS led                          // 이름이 아니라 업체 노드(사업자번호)로 묶는다
        RETURN co.name AS name, led ORDER BY led DESC, name LIMIT 5""", w=work_type, r=region)
    where = f"수요기관에 '{region}'이 든 " if region else ""
    if not found:
        return f"{where}{work_type} 계약에서 주계약업체를 찾지 못했다. {BASIS}"
    return f"{where}{work_type} 주계약업체 상위: " + ", ".join(f"{r['name']}({r['led']}건)" for r in found) + f". {BASIS}"
```


## D04. 라우팅·실행 이력과 실제 StateGraph 구성

38-5의 route는 COUNT_WORDS를 쓰는 단순 기준선이다. '몇 일'을 모두 집계로 보내는 식으로 복제하지 않는다.
아래 원본 전체 셀은 model, tools, SystemMessage, MessagesState, START, ToolNode, tools_condition, StateGraph에 의존한다.
도구 루프와 호출 추적 아이디어는 재사용하되 **v2.0은 고정 Tool registry를 메인이 호출해도 된다. @tool/bind_tools는 필수가 아니다.**
handle_tool_errors는 오류를 메시지로 바꾸는 장치이며, 오류 메시지는 사실 근거가 아니다.
recursion_limit는 LangGraph 단계 수 제한이고, 총 Tool 3회 호출 한도와 같은 값이 아니다.


**수업 원본 — `route`**

`38-5_세-방법-비교와-라우터-벡터RAG-GraphRAG-Text2Cypher_배포용.ipynb`, 전체 셀 34(1부터). SHA-256 `5031a2ffd3877e2c21b18429d78c4f1ec63a65d7027788970973c472106d7fc6`

```python
def route(question):
    return "Text2Cypher" if any(word in question for word in COUNT_WORDS) else "GraphRAG"
```


**수업 원본 — `ToolNode(tools, handle_tool_errors=True)`**

`38-5_세-방법-비교와-라우터-벡터RAG-GraphRAG-Text2Cypher_배포용.ipynb`, 전체 셀 50(1부터). SHA-256 `1761a8477cdf0d1017a13369a18a98ac1b62102093e398032099e602312d1e81`

```python
RULES = """너는 개인정보 보호법 · 시행령 그래프를 안내하는 도우미다. 질문에 맞는 도구를 불러 근거를 찾고, 도구 결과에 있는 내용만으로 답한다.
'법 제N조'는 개인정보 보호법, '영 제N조'는 시행령 조문이다. 답에 근거 조문을 [법 제N조] 모양으로 붙인다.
근거가 모자라면 다른 도구를 더 불러도 된다. 도구 결과에 답이 없으면 '자료에 없음'이라고 답한다."""

model_with_tools = model.bind_tools(tools)

def agent(state: MessagesState):
    response = model_with_tools.invoke([SystemMessage(RULES)] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))    # 도구 오류는 오류 문장을 도구 결과로 모델에게 넘김
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)     # tool_calls 있으면 tools, 없으면 끝
builder.add_edge("tools", "agent")
agent_graph = builder.compile()
print("에이전트 그래프 준비")
```


**수업 원본 — `ask_agent`**

`38-5_세-방법-비교와-라우터-벡터RAG-GraphRAG-Text2Cypher_배포용.ipynb`, 전체 셀 51(1부터). SHA-256 `6ff3d556180ff38c27cd9bbbc5ebb533b25fa44ac0d9532bc63a69e4702d2488`

```python
def ask_agent(question):
    """질문 하나 -> (부른 도구 이름 목록, 마지막 답)"""
    try:      # (참고) recursion_limit 에 걸리면 GraphRecursionError - 그 문항만 '실행 오류'로 남기고 다음 문항으로
        result = agent_graph.invoke({"messages": [HumanMessage(question)]}, {"recursion_limit": 10})
    except Exception as error:
        return ["실행 오류"], f"실행 오류 ({type(error).__name__})"
    calls = [call["name"] for message in result["messages"] for call in (getattr(message, "tool_calls", None) or [])]
    return calls, result["messages"][-1].content
```


## D05. 기존 적재·ID·실패 처리에서 유지할 부분

42-5 rule_link는 by_name·corps·normalize에 의존해 정확 매칭과 NIL 이유를 반환한다.
44-2 run_all은 names·ingest를 호출하고 실패 대상만 모은다. 44-1 gold_keys는 driver/P로 실제 DB 관계 키 집합을 만든다.
표준 ID·원문 근거·오류 격리·실제 조회 키 대조 원리는 유지한다. 오늘은 재적재/재추출이 목적이 아니다.


**수업 원본 — `normalize`**

`42-5_Entity-Linking-회사이름을-공시목록에-연결_배포용.ipynb`, 전체 셀 11(1부터). SHA-256 `d94eb1e084b3efe662fbdf8672f2f416cdfdd4463c02ae4dd9d2f6eec01e1234`

```python
def normalize(name):
    """비교용 이름 - 전각 · 기호를 보통 글자로, 법인 표기 · 공백을 빼고, 영문은 대문자로"""
    name = unicodedata.normalize("NFKC", name)          # （주） -> (주), ㈜ -> (주)
    name = re.sub(r"주식회사|\(주\)|유한회사|\(유\)", "", name)     # '국도화학(주)' -> '국도화학'
    return name.replace(" ", "").upper()                   # 'LS MnM' -> 'LSMNM'
```


**수업 원본 — `rule_link`**

`42-5_Entity-Linking-회사이름을-공시목록에-연결_배포용.ipynb`, 전체 셀 23(1부터). SHA-256 `c6fd1c3bea54370548e1644aa6f9199a179997326f8a3bc1f395663e4d1ba170`

```python
def rule_link(row):
    """규칙으로 고르기 -> (고유번호 또는 '', NIL 이유)"""
    exact = list(by_name.get(normalize(row.mention), []))
    if len(exact) == 1:
        return corps.at[exact[0], "corp_code"], ""
    if len(exact) > 1:
        listed_ones = [i for i in exact if corps.at[i, "상장"]]
        return (corps.at[listed_ones[0], "corp_code"], "") if len(listed_ones) == 1 else ("", "같은 이름 여러 곳")
    return "", "목록에 없음"
```


**수업 원본 — `content_hash`**

`44-2_다시넣어도-안전하게-증분적재-버전-실패재처리_배포용.ipynb`, 전체 셀 20(1부터). SHA-256 `b3d9a411341b77ad305bd81b8f84d99bd8c198108d144b3b49630ff8eb03be4f`

```python
def content_hash(text):
    """본문 해시 - 글자가 하나라도 다르면 다른 값 (앞 16자리만 씀)"""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
```


**수업 원본 — `run_all`**

`44-2_다시넣어도-안전하게-증분적재-버전-실패재처리_배포용.ipynb`, 전체 셀 40(1부터). SHA-256 `b2b675c7e707da2aa6c19dc1df8d0aba78fe507007a688a2a634a7a62181b0d2`

```python
def run_all(targets, schema, schema_version):
    """여러 공시를 차례로 넣고 실패한 접수번호 목록을 반환 - 하나가 실패해도 다음으로 넘어간다"""
    failed = []
    for rcept_no in targets:
        try:                                             # (참고) 실패 처리가 이 날의 주제라 try 를 쓴다
            print(names[rcept_no], rcept_no, "->", ingest(rcept_no, schema, schema_version))
        except Exception as error:
            failed.append(rcept_no)
            print(names[rcept_no], rcept_no, "-> 실패:", type(error).__name__)
    print("실패", len(failed), "건")
    return failed
```


**수업 원본 — `gold_keys`**

`44-1_보고서-KG-Neo4j-적재-한번에_배포용.ipynb`, 전체 셀 48(1부터). SHA-256 `500319ecd9125c988ca069509dbc590bae63f63adb1576982e6d0a893c6665c5`

```python
def gold_keys():
    """지금 그래프의 공급 관계를 (보내는 회사, 받는 회사, 출처 보고서) 집합으로"""
    found = driver.execute_query(f"MATCH (a:{P}GoldCompany)-[r:{P}_GOLD_SUPPLIES]->(b) RETURN a.name AS a, b.name AS b, r.source AS s")
    return {(x["a"], x["b"], x["s"]) for x in found.records}
```


## D06. v2.0 계약 원문과 독립 예시 입력

아래 타입 정의는 동봉한 reference_tool_contract_v2_20261105.md §9 원문이다.
실제 프로젝트에서는 공동 승인된 common 타입을 import한다. 예시 안에서는 원문을 실행하므로 원본 경로 없이 학습할 수 있다.
이 타입만으로 런타임 검증이 되지는 않는다. 다음 셀부터의 ref_ 함수는 새로 작성한 교육 예시다.


In [1]:
from typing import TypedDict, Literal, NotRequired

Region = Literal["NATIONAL", "GYEONGGI"]
ToolName = Literal[
    "action_search", "disease_search",
    "legal_search", "crossdoc_graph_search"
]
ToolStatus = Literal["OK", "PARTIAL", "NO_EVIDENCE", "ERROR"]
AnswerStatus = Literal[
    "ANSWERED", "PARTIAL", "INSUFFICIENT_EVIDENCE", "ERROR"
]
ResponseType = Literal["DOMAIN", "GENERAL", "OUT_OF_SCOPE"]

class QueryInput(TypedDict):
    question: str
    region: Region
    allowed_document_ids: list[str]
    request_id: NotRequired[str]
    task: NotRequired[str]
    preferred_document_ids: NotRequired[list[str]]
    context_entity_ids: NotRequired[list[str]]

class ToolInput(TypedDict):
    question: str
    task: str
    region: Region
    allowed_document_ids: list[str]
    preferred_document_ids: NotRequired[list[str]]
    context_entity_ids: NotRequired[list[str]]
    top_k: NotRequired[int]

class Citation(TypedDict):
    document_id: str
    chunk_id: str
    pdf_page: int
    printed_page: int | None
    excerpt: str

class Finding(TypedDict):
    claim: str
    chunk_ids: list[str]
    action_ids: list[str]
    disease_ids: list[str]
    concept_ids: list[str]

class GraphEdge(TypedDict):
    source_ref: str
    relationship_type: str
    target_ref: str

class GraphPath(TypedDict):
    edges: list[GraphEdge]

class ToolResult(TypedDict):
    tool_name: ToolName
    status: ToolStatus
    findings: list[Finding]
    citations: list[Citation]
    graph_paths: list[GraphPath]
    warnings: list[str]

class QueryOutput(TypedDict):
    response_type: ResponseType
    status: AnswerStatus
    answer: str
    findings: list[Finding]
    citations: list[Citation]
    action_ids: list[str]
    warnings: list[str]
    tools_used: list[ToolName]
    search_rounds: int


In [2]:
from copy import deepcopy

REF_DOCUMENT = 'fixture_school_manual_v1'
REF_CHUNK = f'{REF_DOCUMENT}:p0010:t01r01'
REF_ACTION = f'{REF_DOCUMENT}:p0010:a01'
ref_chunk_store = {
    REF_CHUNK: {'document_id': REF_DOCUMENT, 'chunk_id': REF_CHUNK, 'pdf_page': 10,
                'printed_page': 8, 'text': '[인공 원문] 담당 예시: 담임교사가 연락 항목을 확인한다.'},
}
ref_action_store = {REF_ACTION: {'chunk_ids': [REF_CHUNK], 'role_id': 'HOMEROOM_TEACHER', 'basis': 'EXAMPLE'}}
ref_edge_store = {
    (f'SchoolAction:{REF_ACTION}', 'SUPPORTED_BY', f'Chunk:{REF_CHUNK}'),
    (f'SchoolAction:{REF_ACTION}', 'DONE_BY', 'SchoolRole:HOMEROOM_TEACHER'),
    (f'Document:{REF_DOCUMENT}', 'HAS_CHUNK', f'Chunk:{REF_CHUNK}'),
}
ref_request = {'question': '병명 없이 학교 담임교사의 확인 업무를 알려줘', 'region': 'GYEONGGI',
               'allowed_document_ids': [REF_DOCUMENT], 'request_id': 'fixture-request'}
print('정적 타입·인공 원문 준비; 실제 지침이나 Neo4j 조회 결과가 아님')


정적 타입·인공 원문 준비; 실제 지침이나 Neo4j 조회 결과가 아님


## D07. 실제 조회 레코드 → ToolResult / GraphPath

다음 함수는 메모리 fixture가 DB 레코드를 대신한다. 실제에서는 **반환 레코드에서 확인된 관계만** 직렬화한다.
질문이나 텍스트에 역할명이 있다는 이유로 DONE_BY 경로를 생성하지 않는다.
DB 경로·Citation·주장의 의미는 서로 다른 검사다. GraphPath는 화면에서 역으로 읽더라도 저장 방향을 그대로 보존한다.
기존 v1.2 출력은 ALREADY 검증된 경우만 변환한다. 기존 PARTIAL을 임의 OK로 올리지 않는다.


In [3]:
REF_TOOL_NAMES = {'action_search', 'disease_search', 'legal_search', 'crossdoc_graph_search'}
REF_TOOL_KEYS = {'tool_name', 'status', 'findings', 'citations', 'graph_paths', 'warnings'}
REF_OUTPUT_KEYS = {'response_type', 'status', 'answer', 'findings', 'citations', 'action_ids', 'warnings', 'tools_used', 'search_rounds'}

def ref_input_check(request, *, tool=False):
    required = {'question', 'region', 'allowed_document_ids'} | ({'task'} if tool else set())
    if not isinstance(request, dict) or not required <= request.keys():
        raise ValueError('입력 필수 키 누락')
    if not isinstance(request['question'], str) or not request['question'].strip():
        raise ValueError('빈 질문')
    if request['region'] not in {'NATIONAL', 'GYEONGGI'}:
        raise ValueError('지역 오류')
    allowed = request['allowed_document_ids']
    if not isinstance(allowed, list) or not all(isinstance(x, str) and x for x in allowed):
        raise ValueError('허용 문서 형식 오류')
    for key in ['preferred_document_ids', 'context_entity_ids']:
        if key in request and (not isinstance(request[key], list) or not all(isinstance(x, str) and x for x in request[key])):
            raise ValueError(f'{key} 형식 오류')
    if not set(request.get('preferred_document_ids', [])) <= set(allowed):
        raise ValueError('preferred가 allowed 밖')
    for ref in request.get('context_entity_ids', []):
        label, separator, ident = ref.partition(':')
        if not label or not separator or not ident:
            raise ValueError('타입이 붙은 엔티티 참조 오류')
    if tool and (not isinstance(request['task'], str) or not request['task'].strip()):
        raise ValueError('검색 작업 누락')
    if 'top_k' in request and (type(request['top_k']) is not int or request['top_k'] < 1):
        raise ValueError('top_k 오류')

def ref_empty_tool(name, status, warning):
    return {'tool_name': name, 'status': status, 'findings': [], 'citations': [],
            'graph_paths': [], 'warnings': [warning]}

def ref_path(edges):
    # edges는 실제로 조회된 (source_ref, type, target_ref) 목록이라는 인수 계약이다.
    return {'edges': [{'source_ref': s, 'relationship_type': rel, 'target_ref': t} for s, rel, t in edges]}

def ref_action_search(tool_input):
    ref_input_check(tool_input, tool=True)
    if REF_DOCUMENT not in tool_input['allowed_document_ids']:
        return ref_empty_tool('action_search', 'NO_EVIDENCE', '허용 문서 안에 인공 예시 근거 없음')
    chunk = ref_chunk_store[REF_CHUNK]
    return {'tool_name': 'action_search', 'status': 'OK',
            'findings': [{'claim': '[인공 예시] 담임교사를 연락 확인 담당 예시로 제시한다.',
                          'chunk_ids': [REF_CHUNK], 'action_ids': [REF_ACTION], 'disease_ids': [], 'concept_ids': []}],
            'citations': [{**{k: chunk[k] for k in ['document_id', 'chunk_id', 'pdf_page', 'printed_page']}, 'excerpt': chunk['text']}],
            'graph_paths': [ref_path(sorted(ref_edge_store))], 'warnings': ['공식 지침이 아닌 동작 확인용 fixture']}

def ref_legacy_to_tool(legacy, queried_edges):
    status = {'ANSWERED': 'OK', 'PARTIAL': 'PARTIAL', 'INSUFFICIENT_EVIDENCE': 'NO_EVIDENCE', 'ERROR': 'ERROR'}[legacy['status']]
    empty = status in {'NO_EVIDENCE', 'ERROR'}
    return {'tool_name': 'action_search', 'status': status,
            'findings': [] if empty else deepcopy(legacy['findings']),
            'citations': [] if empty else deepcopy(legacy['citations']),
            'graph_paths': [] if empty or not queried_edges else [ref_path(queried_edges)],
            'warnings': deepcopy(legacy['warnings'])}

ref_tool_input = {k: deepcopy(v) for k, v in ref_request.items() if k != 'request_id'}
ref_tool_input['task'] = '학교 담당과 실제 행동 근거 조회'
ref_tool_result = ref_action_search(ref_tool_input)
assert set(ref_tool_result) == REF_TOOL_KEYS
assert 'answer' not in ref_tool_result and 'domain' not in ref_tool_result
print('Tool 봉투와 정방향 관계 직렬화 예시 확인')


Tool 봉투와 정방향 관계 직렬화 예시 확인


## D08. ToolResult 구조와 원문/경로 진위 대조

수업 D01의 검사와 D05의 키 대조 원리를 v2.0에 적용한다. 다음은 엄격한 fixture 검사 예시다.
실제 검증기는 ID/페이지/허용 문서와 GraphPath를 Neo4j에서 다시 확인해야 한다.
아래 메모리 집합 대조는 실DB 검사 구현이 아니며, 원문 의미의 사람 검수를 대체하지 않는다.


In [4]:
def ref_tool_validate(result, request, expected_name):
    if not isinstance(result, dict) or set(result) != REF_TOOL_KEYS:
        raise ValueError('ToolResult 키 오류')
    if result['tool_name'] not in REF_TOOL_NAMES or result['tool_name'] != expected_name:
        raise ValueError('호출 Tool과 반환 Tool 불일치')
    if result['status'] not in {'OK', 'PARTIAL', 'NO_EVIDENCE', 'ERROR'}:
        raise ValueError('Tool 상태 오류')
    for key in ['findings', 'citations', 'graph_paths', 'warnings']:
        if not isinstance(result[key], list):
            raise ValueError('목록 필드 오류')
    if not all(isinstance(w, str) for w in result['warnings']):
        raise ValueError('경고 형식 오류')
    if result['status'] in {'NO_EVIDENCE', 'ERROR'} and (result['findings'] or result['citations']):
        raise ValueError('빈 결과/오류에 주장 존재')
    if result['status'] in {'OK', 'PARTIAL'} and not result['findings']:
        raise ValueError('성공/부분 결과에 주장 없음')
    if result['status'] != 'OK' and not result['warnings']:
        raise ValueError('제한/오류 사유 누락')
    citations = {}
    for c in result['citations']:
        if set(c) != {'document_id', 'chunk_id', 'pdf_page', 'printed_page', 'excerpt'}:
            raise ValueError('Citation 키 오류')
        known = ref_chunk_store.get(c['chunk_id'])
        if not known or c['chunk_id'] in citations or c['document_id'] not in request['allowed_document_ids']:
            raise ValueError('없는/중복/허용 밖 청크')
        if c['document_id'] != known['document_id'] or type(c['pdf_page']) is not int or c['pdf_page'] != known['pdf_page']:
            raise ValueError('문서 소속/페이지 불일치')
        if c['printed_page'] != known['printed_page'] or (c['printed_page'] is not None and type(c['printed_page']) is not int):
            raise ValueError('인쇄 페이지 불일치')
        if not isinstance(c['excerpt'], str) or not c['excerpt'].strip() or c['excerpt'] not in known['text']:
            raise ValueError('인용 불일치')
        citations[c['chunk_id']] = c
    for f in result['findings']:
        if set(f) != {'claim', 'chunk_ids', 'action_ids', 'disease_ids', 'concept_ids'}:
            raise ValueError('Finding 키 오류')
        if not isinstance(f['claim'], str) or not f['claim'].strip():
            raise ValueError('빈 주장')
        for key in ['chunk_ids', 'action_ids', 'disease_ids', 'concept_ids']:
            if not isinstance(f[key], list) or not all(isinstance(x, str) and x for x in f[key]):
                raise ValueError('Finding 참조 형식 오류')
        if not f['chunk_ids'] or not set(f['chunk_ids']) <= citations.keys():
            raise ValueError('주장 근거 누락')
        for aid in f['action_ids']:
            known = ref_action_store.get(aid)
            if known is None or not set(known['chunk_ids']) & set(f['chunk_ids']):
                raise ValueError('행동 원문 연결 오류')
        # 이 인공 fixture는 질병·개념 노드를 포함하지 않는다.
        if f['disease_ids'] or f['concept_ids']:
            raise ValueError('fixture 사전에 없는 질병/개념')
    for path in result['graph_paths']:
        if set(path) != {'edges'} or not isinstance(path['edges'], list) or not path['edges']:
            raise ValueError('경로 형식 오류')
        for edge in path['edges']:
            if set(edge) != {'source_ref', 'relationship_type', 'target_ref'}:
                raise ValueError('edge 키 오류')
            key = (edge['source_ref'], edge['relationship_type'], edge['target_ref'])
            if key not in ref_edge_store:
                raise ValueError('없는 관계/방향')
            # 참조 ID에 ':'가 있어도 첫 ':'만 타입 구분자로 쓴다.
            for value in [key[0], key[2]]:
                label, _, ident = value.partition(':')
                if label == 'Document' and ident not in request['allowed_document_ids']:
                    raise ValueError('경로의 문서가 허용 밖')
                if label == 'Chunk' and ref_chunk_store[ident]['document_id'] not in request['allowed_document_ids']:
                    raise ValueError('경로의 청크가 허용 밖')
                if label == 'SchoolAction' and any(ref_chunk_store[cid]['document_id'] not in request['allowed_document_ids'] for cid in ref_action_store[ident]['chunk_ids']):
                    raise ValueError('경로의 행동이 허용 밖')

ref_tool_validate(ref_tool_result, ref_tool_input, 'action_search')
print('fixture 원문·ID·관계 대조 통과; 실제 PDF 해석은 별도')


fixture 원문·ID·관계 대조 통과; 실제 PDF 해석은 별도


## D09. 실제 LangGraph를 사용한 11/05의 최소 메인 경로

수업 D04의 StateGraph·조건 분기·호출 추적을 v2.0에 맞춰 적용한 새 예시다.
**규칙 기반 분류는 fixture용 기준선**이다. 실서비스는 실제 지원 질병/행동 사전·골든 질문으로 분류 범위를 검증한다.
병명 유무로 일반 대화 여부를 결정하지 않는다. Tool 미등록은 다른 Tool로 몰래 대체하지 않는다.
총 호출수는 execute_tool에서 실제 callable 호출 때 증가한다. 임베딩/DB 내부 조회수는 별도다.
내일의 평가/루프 확장과 달리 여기서는 한 번 호출 후 종료하고 부족한 부분을 PARTIAL로 남긴다.


In [5]:
from langgraph.graph import StateGraph, START, END

class RefMainState(TypedDict, total=False):
    request: QueryInput
    response_type: str
    intent: str
    selected_tool: str
    tool_result: ToolResult
    tools_used: list[str]
    search_rounds: int
    final_result: QueryOutput

def ref_main_graph(registry):
    def normalize_input(state):
        ref_input_check(state['request'])
        return {'tools_used': [], 'search_rounds': 0}

    def classify_scope(state):
        q = state['request']['question']
        # 일반 안내/지원 밖은 명시한 fixture 문구만 판별한다. 운영용 분류기가 아니다.
        scope = 'GENERAL' if q == '안녕' else 'OUT_OF_SCOPE' if q.startswith('진단 확정') else 'DOMAIN'
        return {'response_type': scope}

    def classify_intent(state):
        q = state['request']['question']
        name = ('crossdoc_graph_search' if '비교' in q else 'legal_search' if q.startswith('신고')
                else 'disease_search' if '잠복기' in q else 'action_search')
        return {'selected_tool': name, 'intent': name}

    def execute_tool(state):
        name = state['selected_tool']
        request = state['request']
        tool_input = {k: deepcopy(request[k]) for k in ['question', 'region', 'allowed_document_ids']}
        for key in ['preferred_document_ids', 'context_entity_ids']:
            if key in request:
                tool_input[key] = deepcopy(request[key])
        tool_input['task'] = '인공 예시: 선택한 검색 기능의 근거 확인'
        fn = registry.get(name)
        if fn is None:
            return {'tool_result': ref_empty_tool(name, 'ERROR', '아직 연결되지 않은 Tool; 대체 호출 없음')}
        # 실제 실행을 시도하므로 실패한 호출도 1회로 기록한다.
        trace = {'tools_used': state['tools_used'] + [name], 'search_rounds': state['search_rounds'] + 1}
        try:
            ref_input_check(tool_input, tool=True)
            result = fn(tool_input)
            ref_tool_validate(result, tool_input, name)
        except Exception as exc:
            result = ref_empty_tool(name, 'ERROR', f'실행/검증 실패: {type(exc).__name__}')
        return {**trace, 'tool_result': result}

    def compose_answer(state):
        scope = state['response_type']
        if scope != 'DOMAIN':
            return {'final_result': {'response_type': scope,
                    'status': 'ANSWERED' if scope == 'GENERAL' else 'INSUFFICIENT_EVIDENCE',
                    'answer': '서비스 안내' if scope == 'GENERAL' else '지원 범위 안내',
                    'findings': [], 'citations': [], 'action_ids': [], 'warnings': [],
                    'tools_used': [], 'search_rounds': 0}}
        result = state['tool_result']
        status = {'OK': 'ANSWERED', 'PARTIAL': 'PARTIAL', 'NO_EVIDENCE': 'INSUFFICIENT_EVIDENCE', 'ERROR': 'ERROR'}[result['status']]
        warnings = list(result['warnings'])
        # 예시: Action 작업은 충분해도 원질문의 법률 부분은 남아 있을 수 있다.
        if result['findings'] and '행동과 신고 기준' in state['request']['question']:
            status = 'PARTIAL'
            warnings.append('학교 행동만 확인; 법률 작업은 추가 확인 필요. 이 예시는 1-Tool 종료 경로다.')
        return {'final_result': {'response_type': 'DOMAIN', 'status': status,
                'answer': '\n'.join(f['claim'] for f in result['findings']) or '확인 가능한 근거 없음 또는 실행 실패',
                'findings': deepcopy(result['findings']), 'citations': deepcopy(result['citations']),
                'action_ids': sorted({aid for f in result['findings'] for aid in f['action_ids']}),
                'warnings': warnings, 'tools_used': state['tools_used'], 'search_rounds': state['search_rounds']}}

    graph = StateGraph(RefMainState)
    for name, fn in [('normalize_input', normalize_input), ('classify_scope', classify_scope),
                     ('classify_intent', classify_intent), ('execute_tool', execute_tool), ('compose_answer', compose_answer)]:
        graph.add_node(name, fn)
    graph.add_edge(START, 'normalize_input')
    graph.add_edge('normalize_input', 'classify_scope')
    graph.add_conditional_edges('classify_scope', lambda s: 'domain' if s['response_type'] == 'DOMAIN' else 'direct',
                               {'domain': 'classify_intent', 'direct': 'compose_answer'})
    graph.add_edge('classify_intent', 'execute_tool')
    graph.add_edge('execute_tool', 'compose_answer')
    graph.add_edge('compose_answer', END)
    return graph.compile()

def ref_run(request, registry):
    # 내부 State를 외부로 반환하지 않고 final_result만 내보낸다.
    ref_input_check(request)
    result = ref_main_graph(registry).invoke({'request': deepcopy(request)})['final_result']
    if set(result) != REF_OUTPUT_KEYS or result['search_rounds'] != len(result['tools_used']):
        raise ValueError('최종 계약/호출 이력 불일치')
    if set(result['action_ids']) != {a for f in result['findings'] for a in f['action_ids']}:
        raise ValueError('행동 합집합 불일치')
    return result

ref_out = ref_run(ref_request, {'action_search': ref_action_search})
assert ref_out['status'] == 'ANSWERED' and ref_out['search_rounds'] == 1
assert ref_out['tools_used'] == ['action_search'] and 'tool_result' not in ref_out
print('실제 StateGraph + 인공 Tool: 1회 호출 후 v2.0 최종 응답')


실제 StateGraph + 인공 Tool: 1회 호출 후 v2.0 최종 응답


## D10. 오류·허용범위·가짜 경로·이행 검사

아래는 실제 LangGraph를 fixture로 실행하는 회귀 검사다. 성공적인 실DB 연결을 증명하는 테스트는 아니다.
ToolResult의 GraphPath는 실제 서비스 최종 QueryOutput에 정의돼 있지 않다. 내부 실행 로그/증빙으로 보존하고
최종 출력에 graph_paths를 임의 추가하지 않는다. T04/T06/T07의 실DB CrossDoc·재검색은 11/06에 이어 검증한다.


In [6]:
def ref_expect_error(fn):
    try:
        fn()
    except ValueError:
        return
    raise AssertionError('거절해야 할 입력이 통과함')

# T02/T05: 병명이 없어도 학교 업무 Tool 하나로 처리.
assert ref_out['tools_used'] == ['action_search']
# 빈 문서 허용 목록: 근거 없음이며 ERROR가 아님.
ref_empty = ref_run({**ref_request, 'allowed_document_ids': []}, {'action_search': ref_action_search})
assert ref_empty['status'] == 'INSUFFICIENT_EVIDENCE' and not ref_empty['findings']
# T08: 가짜 Citation.
ref_forged = deepcopy(ref_tool_result)
ref_forged['citations'][0]['chunk_id'] = 'missing'
ref_expect_error(lambda: ref_tool_validate(ref_forged, ref_tool_input, 'action_search'))
# T09: 실제 방향과 반대인 가짜 경로.
ref_wrong_path = deepcopy(ref_tool_result)
ref_wrong_path['graph_paths'][0]['edges'][0]['relationship_type'] = 'DOES_NOT_EXIST'
ref_expect_error(lambda: ref_tool_validate(ref_wrong_path, ref_tool_input, 'action_search'))
# 다른 Tool인 척 반환하지 못함.
ref_expect_error(lambda: ref_tool_validate(ref_tool_result, ref_tool_input, 'legal_search'))
# 일반 대화·지원 밖은 Tool 0회.
assert ref_run({**ref_request, 'question': '안녕'}, {})['search_rounds'] == 0
assert ref_run({**ref_request, 'question': '진단 확정 요청'}, {})['response_type'] == 'OUT_OF_SCOPE'
# 미등록 Tool은 action_search로 대체하지 않음.
ref_missing = ref_run({**ref_request, 'question': '두 문서를 비교'}, {'action_search': ref_action_search})
assert ref_missing['status'] == 'ERROR' and ref_missing['search_rounds'] == 0
# T11: 실제 호출 시도 후 실패는 1회로 기록.
def ref_broken_tool(request):
    raise ConnectionError('인공 오류')
ref_failed = ref_run(ref_request, {'action_search': ref_broken_tool})
assert ref_failed['status'] == 'ERROR' and ref_failed['search_rounds'] == 1 and not ref_failed['findings']
# Tool OK를 원질문 전체의 완료로 과장하지 않음.
ref_partial = ref_run({**ref_request, 'question': '학교 행동과 신고 기준'}, {'action_search': ref_action_search})
assert ref_partial['status'] == 'PARTIAL'
# 레거시 PARTIAL 유지, 완성 answer/domain 제거.
ref_old = {'domain': 'school', 'status': 'PARTIAL', 'findings': ref_tool_result['findings'],
           'citations': ref_tool_result['citations'], 'warnings': ['검증 범위 제한'], 'answer': '예전 요약'}
assert ref_legacy_to_tool(ref_old, [])['status'] == 'PARTIAL'
# 입력 권한·k 형식 거절.
ref_expect_error(lambda: ref_input_check({**ref_request, 'preferred_document_ids': ['outside']}))
ref_expect_error(lambda: ref_input_check({**ref_tool_input, 'top_k': True}, tool=True))
print('실행/출처/경로/계약 회귀 사례 통과. 실DB·PDF·팀 Tool 3개 단독 호출은 구현 단계에서 별도 증빙.')


실행/출처/경로/계약 회귀 사례 통과. 실DB·PDF·팀 Tool 3개 단독 호출은 구현 단계에서 별도 증빙.


## 실제 구현으로 넘기기

같은 폴더의 `11월05일_메인그래프와_검색도구_부품지시서.md`를 순서대로 따른다.
`11월04일코드_v2_이행점검.md`는 실제 프로젝트 코드를 읽은 재사용/수정 지점이다.
이 예시는 service 코드가 아니며 공동 계약 검증기의 완전한 구현도 아니다.
최종 구현에는 실제 공통 계약·런타임 검사·Neo4j 경로 검사·원문 의미 검증과 허용 문서 통제를 연결한다.
